**Importer les libraries pertinentes**

Source : Prettenhofer, P., Grisel, O., Blondel, M., Amor, A. et Buitinck, L. [Classification of text documents using sparse features](https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html#sphx-glr-auto-examples-text-plot-document-classification-20newsgroups-py). *scikit-learn*. 

In [1]:
import pandas as pd
import os
import re

from IPython.display import clear_output

# Stopwords & punctuation
import string
from nltk.corpus import stopwords

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import multiprocessing
cores = multiprocessing.cpu_count()

# Naive Bayes (GaussianNB)
from sklearn.naive_bayes import GaussianNB

# K plus proches voisins (kNN)
from sklearn.neighbors import KNeighborsClassifier

# Machines à vecteurs de support (SVM)
from sklearn.svm import LinearSVC

# Régression Logistique
from sklearn.linear_model import LogisticRegression

# Forêt aléatoire 
from sklearn.ensemble import RandomForestClassifier

# Perceptron multicouches
from sklearn.neural_network import MLPClassifier

# Validation croisée - 10-fold cross-validation
stratified_kfold = StratifiedKFold(shuffle=True, random_state=42)

**Charger les données nettoyées**

In [2]:
folder = '../2-scripts/1-preprocessing'
datasets = os.listdir(folder)
datasets

['cleaned_train_dataset_10pc.xlsx',
 'cleaned_train_dataset_20pc.xlsx',
 'cleaned_train_dataset_30pc.xlsx',
 'cleaned_train_dataset_40pc.xlsx',
 'cleaned_train_dataset_50pc.xlsx',
 'cleaned_train_dataset_60pc.xlsx',
 'cleaned_train_dataset_70pc.xlsx',
 'cleaned_train_dataset_80pc.xlsx',
 'cleaned_train_dataset_90pc.xlsx']

**Entraîner les modèles et sortir les résultats**

In [3]:
for dataset in datasets:
    ratio = int(dataset[22:-7])
    df = pd.read_excel(os.path.join(folder, dataset))
    report = []    

    corpus = df['cleaned_text_post'].astype('str')
    y = df['category'].astype('str')

    # Valeurs à tester pour le nombre de traits discriminants
    n_features_values = [100, 250, 500, 750, 1000, 2500, 5000, 10000, 15000]

    # Algorithmes à tester
    algorithms = {
        # 'Naive Bayes' : GaussianNB(), 
        # 'kNN (k=1)' : KNeighborsClassifier(n_neighbors=1),
        # 'kNN (k=2)' : KNeighborsClassifier(n_neighbors=2),
        # 'kNN (k=3)' : KNeighborsClassifier(n_neighbors=3),
        # 'Support Vector Machines (SVM)' : LinearSVC(dual="auto"),
        # 'Logistic Regression': LogisticRegression(n_jobs = cores),
        'Random Forest': RandomForestClassifier(n_jobs = cores),
        'Multilayered Perceptron': MLPClassifier(max_iter = 500)
    }

    for n_features in n_features_values:
            vectorizer = TfidfVectorizer(stop_words=stopwords.words('english'), max_features=n_features)
            X = vectorizer.fit_transform(corpus).toarray()

            for algorithm in algorithms.keys():
                # Validation croisée 
                predictions = cross_val_predict(algorithms[algorithm], X, y, cv=stratified_kfold)
                result = classification_report(y, predictions, output_dict=True)

                # Performances (rappel, précision, score F)
                results = {
                    '% Incels' : int(ratio),
                    'Algorithme' : algorithm,
                    'Modèle de vectorisation': 'TF-IDF',
                    'N traits discr.' : n_features,
                    'accuracy': result['accuracy']    
                }
                results.update(result['macro avg'])
                report.append(results)
                
                print(algorithm, f' | {ratio}% Incels | ', f'{n_features} traits discr.\n', classification_report(y, predictions))
                clear_output(wait=True)

    # Exporter les résults
    # support = Nombre d'occurrence dans chaque classe
    report = pd.DataFrame(report)
    report['nb_posts_incels'] = (report['% Incels'].apply(lambda x: x/100) * report['support']).astype(int)
    report['nb_posts_neutral'] = (report['% Incels'].apply(lambda x: 1-(x/100)) * report['support']).astype(int)

    report = report.rename(columns={'support':'nb_posts_total'})
    report['nb_posts_total'] = report['nb_posts_total'].astype(int)

    # Réorganiser l'ordre des colonnes
    report = report[['nb_posts_total', 'nb_posts_incels', 'nb_posts_neutral', '% Incels', 'Algorithme', 
                    'N traits discr.', 'accuracy', 'precision', 'recall','f1-score']]

    report.sort_values(by='f1-score', ascending=False).to_csv(
    f'../3-results/results_training_tfidf_{ratio}pc__RandomForest_MultiLayeredPerceptron.csv', index=False
    )
    

Random Forest  | 40% Incels |  10000 traits discr.
               precision    recall  f1-score   support

       incel       0.75      0.57      0.64     16000
     neutral       0.75      0.87      0.81     24000

    accuracy                           0.75     40000
   macro avg       0.75      0.72      0.72     40000
weighted avg       0.75      0.75      0.74     40000



**Tester le système retenu sur de nouvelles données**

*On reproduit l'apprentissage pour le système retenu uniquement*

In [ ]:
file_train = '../1-data/training_datasets/train_dataset_40pc.xlsx'

df_train = pd.read_excel(file_train)
df_train

,id_post,date_post,subreddit,author,text_post,category
0,fkasskt,2020,ForeverAloneDating,neoquita,Noooooooooo my heart,incel
1,iixpgy7,2022,ForeverAlone,WrothWasp,They're not really mad at you. They're mad at ...,incel
2,iy64ndj,2022,ForeverAlone,Revelc69,Do you think there are other forces at work as...,incel
3,de0uica,2017,Incels,dannymason,"""The thought of suicide is a powerful solace: ...",incel
4,h1m2nof,2021,ForeverAlone,ToiletPaperGanon,"I hate to sound judgemental, but some of them ...",incel
...,...,...,...,...,...,...
39995,hv2csmy,2022,xbox360,SketchyEntertainment,I'd be down to play anything if you got discord.,neutral
39996,hewyntt,2021,CryptoCurrency,1078Garage,![gif](giphy|IXPmispmx6stWHaftT|downsized),neutral
39997,hv2cfkc,2022,MMA,Ryzentek,"100 percent, they even mentioned how he's a ""g...",neutral
39998,hv2cu81,2022,brasilivre,[deleted],"Eu juro que eu sou incel, por favor não me den...",neutral


In [ ]:
# Système retenu : Naive Bayes ; 40% de données Incels ; 5 000 traits discriminants
X_train = df_train['text_post'].astype(str)
y_train = df_train['category']

ngram_values = (1, 2)

# Antidictionnaire personnalisé
with open('./utils/exclusion_v2.stop', encoding='utf-8') as f:
    custom_stopwords = [x.lower().strip('\n') for x in f.readlines()]
    custom_stopwords += list(ENGLISH_STOP_WORDS)

pipeline_nb = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=custom_stopwords, ngram_range=ngram_values, max_features=15000)),
    ('nb', MultinomialNB())
])

pipeline_nb.fit(X_train, y_train)

NameError: name 'ENGLISH_STOP_WORDS' is not defined

In [ ]:
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nb_predictions = cross_val_predict(pipeline_nb, X_train, y_train, cv=stratified_kfold)

# Performances en phase d'apprentissage du modèle retenu (rappel, précision, score F)
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_train, nb_predictions)
precision_nb = precision_score(y_train, nb_predictions, average='macro')
recall_nb = recall_score(y_train, nb_predictions, average='macro')
f1_nb = f1_score(y_train, nb_predictions, average='macro')

df = [
    {
        'Phase': 'Apprentissage',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb
    }
]

print(classification_report(y_train, nb_predictions))

*On teste sur de nouvelles données*

In [ ]:
file_test = '../1-data/test_dataset_10pc.xlsx'

df_test = pd.read_excel(file_test)
df_test

*Vérifier qu'aucune donnée test n'était contenue dans les données d'appentissage*

In [ ]:
validate = pd.concat([df_train, df_test])

validate[validate.duplicated(subset='id_post')]

*Appliquer le classifieur aux nouvelles données*

In [ ]:
X_test = df_test['text_post'].astype(str)
y_test = df_test['category']

y_pred_nb = pipeline_nb.predict(X_test)

In [ ]:
# Calculer et afficher les performances
accuracy_nb = accuracy_score(y_test, y_pred_nb)
precision_nb = precision_score(y_test, y_pred_nb, average='macro')
recall_nb = recall_score(y_test, y_pred_nb, average='macro')
f1_nb = f1_score(y_test, y_pred_nb, average='macro')

df.append(
    {
        'Phase': 'Test',
        'Précision': precision_nb,
        'Rappel': recall_nb,
        'Mesure F': f1_nb,
        'Exactitude': accuracy_nb

    }
)

pd.set_option("display.precision", 4)
df = pd.DataFrame(df)
df.to_csv('../3-results/results_test.csv', index=False)
df

In [ ]:
accuracy_nb

In [ ]:
precision_nb

In [ ]:
recall_nb

In [ ]:
f1_nb

In [ ]:
print(classification_report(y_test, y_pred_nb))

In [ ]:
import pandas as pd
results = pd.read_csv('../3-results/results_training.csv')
results.sort_values(by='f_score', ascending=False)